# CHAPTER 7 Data Cleaning and Preparation

En este capitulo se discuten herramientas para tratar con data perdida, duplicada, manipuilación de String y otras transformaciones analitica de data.

## 7.1 Handling Missing Data

In [1]:
# Comandos para que Jupyter evalúe y muestre todas las expresiones de una celda automáticamente
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import numpy as np
import pandas as pd
PREVIOUS_MAX_ROWS = pd.options.display.max_rows
pd.options.display.max_rows = 25
pd.options.display.max_columns = 20
pd.options.display.max_colwidth = 82
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc("figure", figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

In [2]:
import numpy as np
import pandas as pd

In [3]:
float_data = pd.Series([1.2, -3.5, np.nan, 0])
float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

In [4]:
# The isna method gives us a Boolean Series with True where values are null
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

**The built-in Python None value is also treated as NA**

In [5]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])
string_data
string_data.isna()
float_data = pd.Series([1, 2, None], dtype='float64')
float_data
float_data.isna()

0    aardvark
1         NaN
2        None
3     avocado
dtype: object

0    False
1     True
2     True
3    False
dtype: bool

0    1.0
1    2.0
2    NaN
dtype: float64

0    False
1    False
2     True
dtype: bool

### Table 7-1. NA handling object methods

| Method | Description |
| :--- | :--- |
| `dropna` | Filtrar las etiquetas de los ejes en función de si los valores de cada etiqueta contienen datos faltantes, con diferentes umbrales para la cantidad de datos faltantes a tolerar. |
| `fillna` | Rellenar los datos faltantes con algún valor o utilizando un método de interpolación como `"ffill"` o `"bfill"`. |
| `isna` | Devolver valores booleanos que indican qué valores son faltantes/NA. |
| `notna` | Negación de `isna`, devuelve `True` para valores que no son NA y `False` para valores NA. |

## Filtering Out Missing Data
There are a few ways to filter out missing data. 

In [7]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])
data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

In [8]:
data[data.notna()]

0    1.0
2    3.5
4    7.0
dtype: float64

In [10]:
data = pd.DataFrame([[1., 6.5, 3.], [1., np.nan, np.nan],
                     [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])
data
data.dropna() # Quita el registro en donde haya al menos un registro vacío

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


,0,1,2
0,1.0,6.5,3.0


In [11]:
# Passing how="all" will drop only rows that are all NA:
data.dropna(how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


In [12]:
# To drop columns in the same way, pass axis="columns"
data[4] = np.nan
data
data.dropna(axis="columns", how="all")

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [13]:
# Suppose you want to keep only rows containing at most a certain number of missing
# observations. You can indicate this with the thresh argument:
df = pd.DataFrame(np.random.standard_normal((7, 3)))
df.iloc[:4, 1] = np.nan
df.iloc[:2, 2] = np.nan
df
df.dropna()
df.dropna(thresh=2)

,0,1,2
0,-0.204708,NaN,NaN
1,-0.555730,NaN,NaN
2,0.092908,NaN,0.769023
3,1.246435,NaN,-1.296221
4,0.274992,0.228913,1.352917
5,0.886429,-2.001637,-0.371843
6,1.669025,-0.438570,-0.539741


,0,1,2
4,0.274992,0.228913,1.352917
5,0.886429,-2.001637,-0.371843
6,1.669025,-0.438570,-0.539741


,0,1,2
2,0.092908,NaN,0.769023
3,1.246435,NaN,-1.296221
4,0.274992,0.228913,1.352917
5,0.886429,-2.001637,-0.371843
6,1.669025,-0.438570,-0.539741


In [14]:
df.fillna(0)

,0,1,2
0,-0.204708,0.000000,0.000000
1,-0.555730,0.000000,0.000000
2,0.092908,0.000000,0.769023
3,1.246435,0.000000,-1.296221
4,0.274992,0.228913,1.352917
5,0.886429,-2.001637,-0.371843
6,1.669025,-0.438570,-0.539741


In [15]:
# Calling fillna with a dictionary, you can use a different fill value for each column:
df.fillna({1: 0.5, 2: 0})

,0,1,2
0,-0.204708,0.500000,0.000000
1,-0.555730,0.500000,0.000000
2,0.092908,0.500000,0.769023
3,1.246435,0.500000,-1.296221
4,0.274992,0.228913,1.352917
5,0.886429,-2.001637,-0.371843
6,1.669025,-0.438570,-0.539741


In [16]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))
df.iloc[2:, 1] = np.nan
df.iloc[4:, 2] = np.nan
df
df.fillna(method="ffill")
df.fillna(method="ffill", limit=2)

,0,1,2
0,0.476985,3.248944,-1.021228
1,-0.577087,0.124121,0.302614
2,0.523772,NaN,1.343810
3,-0.713544,NaN,-2.370232
4,-1.860761,NaN,NaN
5,-1.265934,NaN,NaN


C:\Users\ferch\AppData\Local\Temp\ipykernel_7288\1607676785.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill")


,0,1,2
0,0.476985,3.248944,-1.021228
1,-0.577087,0.124121,0.302614
2,0.523772,0.124121,1.343810
3,-0.713544,0.124121,-2.370232
4,-1.860761,0.124121,-2.370232
5,-1.265934,0.124121,-2.370232


C:\Users\ferch\AppData\Local\Temp\ipykernel_7288\1607676785.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", limit=2)


,0,1,2
0,0.476985,3.248944,-1.021228
1,-0.577087,0.124121,0.302614
2,0.523772,0.124121,1.343810
3,-0.713544,0.124121,-2.370232
4,-1.860761,NaN,-2.370232
5,-1.265934,NaN,-2.370232


In [17]:
data = pd.Series([1., np.nan, 3.5, np.nan, 7])
data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

### Table 7-2. fillna function arguments

| Argument | Description |
| :--- | :--- |
| `value` | Valor escalar o un objeto similar a un diccionario para usar al rellenar los valores faltantes. |
| `method` | Método de interpolación: uno de `"bfill"` (relleno hacia atrás) o `"ffill"` (relleno hacia adelante); por defecto es `None`. |
| `axis` | Eje sobre el cual rellenar (`"index"` o `"columns"`); por defecto es `axis="index"`. |
| `limit` | Para el relleno hacia adelante y hacia atrás, número máximo de períodos consecutivos a rellenar. |

## 7.2 Data Transformation

### Removing Duplicates

In [18]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


In [20]:
# The DataFrame method duplicated returns a Boolean Series indicating whether each row is a duplicate 
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

In [21]:
# En este sentido, drop_duplicates devuelve un DataFrame del que se han eliminado las filas en las que el valor del campo «duplicated» es False.
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Both methods by default consider all of the columns; alternatively, you can specify any subset of them to detect duplicates. Suppose we had an additional column of values and wanted to filter duplicates based only on the "k1" column.

duplicated and drop_duplicates by default keep the first observed value combination. Passing keep="last" will return the last one

In [22]:
data["v1"] = range(7)
data
data.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


,k1,k2,v1
0,one,1,0
1,two,1,1


In [23]:
data.drop_duplicates(["k1", "k2"], keep="last")

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## Transforming Data Using a Function or Mapping

For many datasets, you may wish to perform some transformation based on the values in an array, Series, or column in a DataFrame.

In [24]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


In [25]:
meat_to_animal = {
  "bacon": "pig",
  "pulled pork": "pig",
  "pastrami": "cow",
  "corned beef": "cow",
  "honey ham": "pig",
  "nova lox": "salmon"
}

In [26]:
data["animal"] = data["food"].map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [27]:
def get_animal(x):
    return meat_to_animal[x]
data["food"].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

### Replacing Values
Filling in missing data with the fillna method is a special case of more general value replacement. As you’ve already seen, map can be used to modify a subset of values in an object, but replace provides a simpler and more flexible way to do so. 

In [28]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])
data

0       1.0
1    -999.0
2       2.0
3    -999.0
4   -1000.0
5       3.0
dtype: float64

In [29]:
data.replace(-999, np.nan)

0       1.0
1       NaN
2       2.0
3       NaN
4   -1000.0
5       3.0
dtype: float64

In [30]:
# Reemplazar multiples valores a la vez
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

In [31]:
# remplazar cada valor por otro independientes
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

In [32]:
# también se puede hacer el cambio independiente en forma de diccionario
data.replace({-999: np.nan, -1000: 0})

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

### Renaming Axis Indexes

Like values in a Series, axis labels can be similarly transformed by a function or mapping of some form to produce new, differently labeled objects. You can also modify the axes in place without creating a new data structure. 

In [34]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


In [35]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

In [36]:
data.index = data.index.map(transform)
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


In [37]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


In [38]:
data.rename(index={"OHIO": "INDIANA"},
            columns={"three": "peekaboo"})

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


### Discretization and Binning
Continuous data is often discretized or otherwise separated into “bins” for analysis. Suppose you have data about a group of people in a study, and you want to group them into discrete age buckets:

In [39]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

In [40]:
bins = [18, 25, 35, 60, 100]
age_categories = pd.cut(ages, bins)
age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

In [41]:
age_categories.codes
age_categories.categories
age_categories.categories[0]
pd.value_counts(age_categories)

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

Interval(18, 25, closed='right')

C:\Users\ferch\AppData\Local\Temp\ipykernel_7288\4232757300.py:4: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(age_categories)


(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

In [42]:
# esto para que el conjunto no contenga el valor a la derecha del subset
pd.cut(ages, bins, right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

In [44]:
# se pueden sobreescribir el label de la categoria por uno de la preferencia
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]
pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

In [45]:
data = np.random.uniform(size=20)
pd.cut(data, 4, precision=2)

[(0.34, 0.55], (0.34, 0.55], (0.76, 0.97], (0.76, 0.97], (0.34, 0.55], ..., (0.34, 0.55], (0.34, 0.55], (0.55, 0.76], (0.34, 0.55], (0.12, 0.34]]
Length: 20
Categories (4, interval[float64, right]): [(0.12, 0.34] < (0.34, 0.55] < (0.55, 0.76] < (0.76, 0.97]]

In [46]:
data = np.random.standard_normal(1000)
quartiles = pd.qcut(data, 4, precision=2)
quartiles
pd.value_counts(quartiles)

[(-0.026, 0.62], (0.62, 3.93], (-0.68, -0.026], (0.62, 3.93], (-0.026, 0.62], ..., (-0.68, -0.026], (-0.68, -0.026], (-2.96, -0.68], (0.62, 3.93], (-0.68, -0.026]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.96, -0.68] < (-0.68, -0.026] < (-0.026, 0.62] < (0.62, 3.93]]

C:\Users\ferch\AppData\Local\Temp\ipykernel_7288\3037377620.py:4: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(quartiles)


(-2.96, -0.68]     250
(-0.68, -0.026]    250
(-0.026, 0.62]     250
(0.62, 3.93]       250
Name: count, dtype: int64

In [47]:
pd.qcut(data, [0, 0.1, 0.5, 0.9, 1.]).value_counts()

(-2.9499999999999997, -1.187]    100
(-1.187, -0.0265]                400
(-0.0265, 1.286]                 400
(1.286, 3.928]                   100
Name: count, dtype: int64

In [48]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.049091,0.026112,-0.002544,-0.051827
std,0.996947,1.007458,0.995232,0.998311
min,-3.645860,-3.184377,-3.745356,-3.428254
25%,-0.599807,-0.612162,-0.687373,-0.747478
50%,0.047101,-0.013609,-0.022158,-0.088274
75%,0.756646,0.695298,0.699046,0.623331
max,2.653656,3.525865,2.735527,3.366626


### Detecting and Filtering Outliers
Filtering or transforming outliers is largely a matter of applying array operations. Suppose you wanted to find values in one of the columns exceeding 3 in absolute value:

In [49]:
col = data[2]
col[col.abs() > 3]

41    -3.399312
136   -3.745356
Name: 2, dtype: float64

In [50]:
data[(data.abs() > 3).any(axis="columns")]

,0,1,2,3
41,0.457246,-0.025907,-3.399312,-0.974657
60,1.951312,3.260383,0.963301,1.201206
136,0.508391,-0.196713,-3.745356,-1.520113
235,-0.242459,-3.056990,1.918403,-0.578828
258,0.682841,0.326045,0.425384,-3.428254
322,1.179227,-3.184377,1.369891,-1.074833
544,-3.548824,1.553205,-2.186301,1.277104
635,-0.578093,0.193299,1.397822,3.366626
782,-0.207434,3.525865,0.283070,0.544635
803,-3.645860,0.255475,-0.549574,-1.907459


In [51]:
# The statement np.sign(data) produces 1 and –1 values based on whether the values
# in data are positive or negative
data[data.abs() > 3] = np.sign(data) * 3
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.050286,0.025567,-0.001399,-0.051765
std,0.992920,1.004214,0.991414,0.995761
min,-3.000000,-3.000000,-3.000000,-3.000000
25%,-0.599807,-0.612162,-0.687373,-0.747478
50%,0.047101,-0.013609,-0.022158,-0.088274
75%,0.756646,0.695298,0.699046,0.623331
max,2.653656,3.000000,2.735527,3.000000


In [53]:
data.head()
np.sign(data).head()

,0,1,2,3
0,-0.799318,0.777233,-0.612905,0.316447
1,0.838295,-1.034423,0.434304,-2.213133
2,0.758040,0.553933,0.339231,-0.688756
3,-0.815526,-0.332420,2.406483,-1.361428
4,-0.669619,0.781199,-0.395813,-0.180737


,0,1,2,3
0,-1.0,1.0,-1.0,1.0
1,1.0,-1.0,1.0,-1.0
2,1.0,1.0,1.0,-1.0
3,-1.0,-1.0,1.0,-1.0
4,-1.0,1.0,-1.0,-1.0


### Permutation and Random Sampling
Permuting (randomly reordering) a Series or the rows in a DataFrame is possible using the numpy.random.permutation function. Calling permutation with the length of the axis you want to permute produces an array of integers indicating the new ordering

In [54]:
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))
df
sampler = np.random.permutation(5)
sampler

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


array([3, 1, 4, 2, 0], dtype=int32)

In [55]:
df.take(sampler)
df.iloc[sampler]

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13
4,28,29,30,31,32,33,34
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6


,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13
4,28,29,30,31,32,33,34
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6


In [56]:
# By invoking take with axis="columns", we could also select a permutation of the columns
column_sampler = np.random.permutation(7)
column_sampler
df.take(column_sampler, axis="columns")

array([4, 6, 3, 2, 1, 0, 5], dtype=int32)

,4,6,3,2,1,0,5
0,4,6,3,2,1,0,5
1,11,13,10,9,8,7,12
2,18,20,17,16,15,14,19
3,25,27,24,23,22,21,26
4,32,34,31,30,29,28,33


In [57]:
# para seleccionar un subset random sin remplazo
df.sample(n=3)

,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
4,28,29,30,31,32,33,34
0,0,1,2,3,4,5,6


In [58]:
# para generar un ejemplo con reemplazo
choices = pd.Series([5, 7, -1, 6, 4])
choices.sample(n=10, replace=True)

2   -1
0    5
3    6
1    7
4    4
0    5
4    4
0    5
4    4
4    4
dtype: int64

In [59]:
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                   "data1": range(6)})
df
pd.get_dummies(df["key"], dtype=float)

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


,a,b,c
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
5,0.0,1.0,0.0


In [60]:
dummies = pd.get_dummies(df["key"], prefix="key", dtype=float)
df_with_dummy = df[["data1"]].join(dummies)
df_with_dummy

,data1,key_a,key_b,key_c
0,0,0.0,1.0,0.0
1,1,0.0,1.0,0.0
2,2,1.0,0.0,0.0
3,3,0.0,0.0,1.0
4,4,1.0,0.0,0.0
5,5,0.0,1.0,0.0


In [61]:
mnames = ["movie_id", "title", "genres"]
movies = pd.read_table("datasets/movielens/movies.dat", sep="::",
                       header=None, names=mnames, engine="python")
movies[:10]

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children's
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [62]:
dummies = movies["genres"].str.get_dummies("|")
dummies.iloc[:10, :6]

,Action,Adventure,Animation,Children's,Comedy,Crime
0,0,0,1,1,1,0
1,0,1,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,1,0
4,0,0,0,0,1,0
5,1,0,0,0,0,1
6,0,0,0,0,1,0
7,0,1,0,1,0,0
8,1,0,0,0,0,0
9,1,1,0,0,0,0


In [63]:
movies_windic = movies.join(dummies.add_prefix("Genre_"))
movies_windic.iloc[0]

movie_id                                       1
title                           Toy Story (1995)
genres               Animation|Children's|Comedy
Genre_Action                                   0
Genre_Adventure                                0
Genre_Animation                                1
Genre_Children's                               1
Genre_Comedy                                   1
Genre_Crime                                    0
Genre_Documentary                              0
Genre_Drama                                    0
Genre_Fantasy                                  0
Genre_Film-Noir                                0
Genre_Horror                                   0
Genre_Musical                                  0
Genre_Mystery                                  0
Genre_Romance                                  0
Genre_Sci-Fi                                   0
Genre_Thriller                                 0
Genre_War                                      0
Genre_Western       

In [64]:
np.random.seed(12345) # to make the example repeatable
values = np.random.uniform(size=10)
values
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]
pd.get_dummies(pd.cut(values, bins))

array([0.9296, 0.3164, 0.1839, 0.2046, 0.5677, 0.5955, 0.9645, 0.6532,
       0.7489, 0.6536])

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False


## 7.3 Extension Data Types
More recently, pandas has developed an extension type system allowing for new data types to be added even if they are not supported natively by NumPy. These new data types can be treated as first class alongside data coming from NumPy arrays.

In [65]:
s = pd.Series([1, 2, 3, None])
s
s.dtype

0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64

dtype('float64')

**The output \<NA\> indicates that a value is missing for an extension type array. This
uses the special *pandas.NA* sentinel value:**

In [66]:
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())
s
s.isna()
s.dtype

0       1
1       2
2       3
3    <NA>
dtype: Int64

0    False
1    False
2    False
3     True
dtype: bool

Int64Dtype()

In [67]:
s[3]
s[3] is pd.NA

<NA>

True

In [68]:
s = pd.Series([1, 2, 3, None], dtype="Int64")

In [69]:
s = pd.Series(['one', 'two', None, 'three'], dtype=pd.StringDtype())
s

0      one
1      two
2     <NA>
3    three
dtype: string

In [70]:
df = pd.DataFrame({"A": [1, 2, None, 4],
                   "B": ["one", "two", "three", None],
                   "C": [False, None, False, True]})
df
df["A"] = df["A"].astype("Int64")
df["B"] = df["B"].astype("string")
df["C"] = df["C"].astype("boolean")
df

,A,B,C
0,1.0,one,False
1,2.0,two,None
2,NaN,three,False
3,4.0,None,True


,A,B,C
0,1,one,False
1,2,two,<NA>
2,<NA>,three,False
3,4,<NA>,True


### Table 7-3. pandas extension data types

| Extension type | Description |
| :--- | :--- |
| `BooleanDtype` | Datos booleanos que admiten valores nulos (nullable), use `"boolean"` al pasarlo como cadena. |
| `CategoricalDtype` | Tipo de dato categórico, use `"category"` al pasarlo como cadena. |
| `DatetimeTZDtype` | Fecha y hora con zona horaria. |
| `Float32Dtype` | Punto flotante de 32 bits que admite valores nulos, use `"Float32"` al pasarlo como cadena. |
| `Float64Dtype` | Punto flotante de 64 bits que admite valores nulos, use `"Float64"` al pasarlo como cadena. |
| `Int8Dtype` | Entero con signo de 8 bits que admite valores nulos, use `"Int8"` al pasarlo como cadena. |
| `Int16Dtype` | Entero con signo de 16 bits que admite valores nulos, use `"Int16"` al pasarlo como cadena. |
| `Int32Dtype` | Entero con signo de 32 bits que admite valores nulos, use `"Int32"` al pasarlo como cadena. |
| `Int64Dtype` | Entero con signo de 64 bits que admite valores nulos, use `"Int64"` al pasarlo como cadena. |
| `UInt8Dtype` | Entero sin signo de 8 bits que admite valores nulos, use `"UInt8"` al pasarlo como cadena. |
| `UInt16Dtype` | Entero sin signo de 16 bits que admite valores nulos, use `"UInt16"` al pasarlo como cadena. |
| `UInt32Dtype` | Entero sin signo de 32 bits que admite valores nulos, use `"UInt32"` al pasarlo como cadena. |
| `UInt64Dtype` | Entero sin signo de 64 bits que admite valores nulos, use `"UInt64"` al pasarlo como cadena. |

## 7.4 String Manipulation
Most text operations are made simple with the string object’s built-in methods. For more complex pattern matching and
text manipulations, regular expressions may be needed. pandas adds to the mix by enabling you to apply string and regular expressions concisely on whole arrays of data, additionally handling the annoyance of missing data.

### Python Built-In String Object Methods


In [71]:
val = "a,b,  guido"
val.split(",")

['a', 'b', '  guido']

In [72]:
# split is often combined with strip to trim whitespace (including line breaks)
pieces = [x.strip() for x in val.split(",")]
pieces

['a', 'b', 'guido']

In [73]:
# concatenación
first, second, third = pieces
first + "::" + second + "::" + third

'a::b::guido'

In [74]:
# A faster and more Pythonic way is to pass a list or tuple to the join method on the string
"::".join(pieces)

'a::b::guido'

In [75]:
"guido" in val
val.index(",")
val.find(":")

True

1

-1

In [76]:
# raise a exception
val.index(":")

ValueError: substring not found

In [77]:
# devuelve el numero de ocurrencias del substring
val.count(",")

2

In [78]:
val.replace(",", "::")
val.replace(",", "")

'a::b::  guido'

'ab  guido'

### Table 7-4. Python built-in string methods

| Method | Description |
| :--- | :--- |
| `count` | Devuelve el número de ocurrencias no superpuestas de una subcadena en la cadena. |
| `endswith` | Devuelve `True` si la cadena termina con el sufijo especificado. |
| `startswith` | Devuelve `True` si la cadena comienza con el prefijo especificado. |
| `join` | Utiliza la cadena como delimitador para concatenar una secuencia de otras cadenas. |
| `index` | Devuelve el índice inicial de la primera ocurrencia de la subcadena pasada si se encuentra en la cadena; de lo contrario, genera un `ValueError`. |
| `find` | Devuelve la posición del primer carácter de la *primera* ocurrencia de la subcadena en la cadena; es como `index`, pero devuelve `-1` si no se encuentra. |
| `rfind` | Devuelve la posición del primer carácter de la *última* ocurrencia de la subcadena en la cadena; devuelve `-1` si no se encuentra. |
| `replace` | Reemplaza las ocurrencias de una cadena con otra cadena. |
| `strip`, `rstrip`, `lstrip` | Recorta los espacios en blanco, incluidos los saltos de línea, en ambos lados, en el lado derecho o en el lado izquierdo, respectivamente. |
| `split` | Divide la cadena en una lista de subcadenas utilizando el delimitador pasado. |
| `lower` | Convierte los caracteres del alfabeto a minúsculas. |
| `upper` | Convierte los caracteres del alfabeto a mayúsculas. |
| `casefold` | Convierte los caracteres a minúsculas y elimina cualquier combinación de caracteres variables específica de una región para obtener una forma común comparable. |
| `ljust`, `rjust` | Justifica a la izquierda o a la derecha, respectivamente; rellena el lado opuesto de la cadena con espacios (o algún otro carácter de relleno) para devolver una cadena con un ancho mínimo. |

## Regular Expressions
The **re module** functions fall into three categories: pattern matching, substitution, and splitting. Naturally these are all related; a regex describes a pattern to locate in the text, which can then be used for many purposes. Let’s look at a simple example: suppose we wanted to split a string with a variable number of whitespace characters
(tabs, spaces, and newlines).

In [79]:
import re
text = "foo    bar\t baz  \tqux"
re.split(r"\s+", text)

['foo', 'bar', 'baz', 'qux']

In [80]:
# You can compile the regex yourself with re.compile, forming a reusable regex object:
regex = re.compile(r"\s+")
regex.split(text)

['foo', 'bar', 'baz', 'qux']

In [81]:
regex.findall(text)

['    ', '\t ', '  \t']

In [82]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""
pattern = r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}"

# re.IGNORECASE makes the regex case insensitive
regex = re.compile(pattern, flags=re.IGNORECASE)

In [83]:
regex.findall(text)

['dave@google.com', 'steve@gmail.com', 'rob@gmail.com', 'ryan@yahoo.com']

In [84]:
m = regex.search(text)
m
text[m.start():m.end()]

<re.Match object; span=(5, 20), match='dave@google.com'>

'dave@google.com'

In [85]:
# regex.match returns None, as it will match only if the pattern occurs at the start of the string:
print(regex.match(text))

None


In [86]:
# Relatedly, sub will return a new string with occurrences of the pattern replaced by a new string
print(regex.sub("REDACTED", text))

Dave REDACTED
Steve REDACTED
Rob REDACTED
Ryan REDACTED


In [87]:
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
regex = re.compile(pattern, flags=re.IGNORECASE)

In [88]:
m = regex.match("wesm@bright.net")
m.groups()

('wesm', 'bright', 'net')

In [89]:
regex.findall(text)

[('dave', 'google', 'com'),
 ('steve', 'gmail', 'com'),
 ('rob', 'gmail', 'com'),
 ('ryan', 'yahoo', 'com')]

In [92]:
#sub also has access to groups in each match using special symbols like \1 and \2. The
#symbol \1 corresponds to the first matched group, \2 corresponds to the second, and
#so forth:
print(regex.sub(r"Username: \1, Domain: \2, Suffix: \3", text))

Dave Username: dave, Domain: google, Suffix: com
Steve Username: steve, Domain: gmail, Suffix: com
Rob Username: rob, Domain: gmail, Suffix: com
Ryan Username: ryan, Domain: yahoo, Suffix: com


### Table 7-5. Regular expression methods

| Method | Description |
| :--- | :--- |
| `findall` | Devuelve en una lista todos los patrones coincidentes que no se superpongan en una cadena. |
| `finditer` | Similar a `findall`, pero devuelve un iterador. |
| `match` | Busca un patrón al comienzo de la cadena y, opcionalmente, segmenta los componentes del patrón en grupos; si el patrón coincide, devuelve un objeto de coincidencia (match object), de lo contrario devuelve `None`. |
| `search` | Escanea la cadena en busca de una coincidencia con el patrón, devolviendo un objeto de coincidencia en caso de encontrarlo; a diferencia de `match`, la coincidencia puede estar en cualquier lugar de la cadena en lugar de limitarse solo al principio. |
| `split` | Divide la cadena en fragmentos en cada ocurrencia del patrón. |
| `sub`, `subn` | Reemplaza todas (`sub`) o las primeras n ocurrencias (`subn`) del patrón en la cadena con una expresión de reemplazo; utiliza los símbolos `\1`, `\2`, ... para hacer referencia a los elementos de los grupos de coincidencia en la cadena de reemplazo. |

### String Functions in pandas

In [93]:
data = {"Dave": "dave@google.com", "Steve": "steve@gmail.com",
        "Rob": "rob@gmail.com", "Wes": np.nan}
data = pd.Series(data)
data
data.isna()

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                  NaN
dtype: object

Dave     False
Steve    False
Rob      False
Wes       True
dtype: bool

In [94]:
data.str.contains("gmail")

Dave     False
Steve     True
Rob       True
Wes        NaN
dtype: object

In [95]:
data_as_string_ext = data.astype('string')
data_as_string_ext
data_as_string_ext.str.contains("gmail")

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                 <NA>
dtype: string

Dave     False
Steve     True
Rob       True
Wes       <NA>
dtype: boolean

In [96]:
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
data.str.findall(pattern, flags=re.IGNORECASE)

Dave     [(dave, google, com)]
Steve    [(steve, gmail, com)]
Rob        [(rob, gmail, com)]
Wes                        NaN
dtype: object

In [97]:
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]
matches
matches.str.get(1)

Dave     (dave, google, com)
Steve    (steve, gmail, com)
Rob        (rob, gmail, com)
Wes                      NaN
dtype: object

Dave     google
Steve     gmail
Rob       gmail
Wes         NaN
dtype: object

In [100]:
data.str[:5]

Dave     dave@
Steve    steve
Rob      rob@g
Wes        NaN
dtype: object

In [99]:
# The str.extract method will return the captured groups of a regular expression as a DataFrame
data.str.extract(pattern, flags=re.IGNORECASE)

,0,1,2
Dave,dave,google,com
Steve,steve,gmail,com
Rob,rob,gmail,com
Wes,NaN,NaN,NaN


### Table 7-6. Partial listing of Series string methods

| Method | Description |
| :--- | :--- |
| `cat` | Concatenar cadenas elemento por elemento con un delimitador opcional. |
| `contains` | Devuelve un arreglo booleano si cada cadena contiene el patrón/expresión regular. |
| `count` | Contar las ocurrencias de un patrón. |
| `extract` | Utiliza una expresión regular con grupos para extraer una o más cadenas de una Serie de cadenas; el resultado será un DataFrame con una columna por grupo. |
| `endswith` | Equivalente a `x.endswith(pattern)` para cada elemento. |
| `startswith` | Equivalente a `x.startswith(pattern)` para cada elemento. |
| `findall` | Calcula la lista de todas las ocurrencias del patrón/expresión regular para cada cadena. |
| `get` | Indexa dentro de cada elemento (recupera el i-ésimo elemento). |
| `isalnum` | Equivalente a la función integrada `str.isalnum`. |
| `isalpha` | Equivalente a la función integrada `str.isalpha`. |
| `isdecimal` | Equivalente a la función integrada `str.isdecimal`. |
| `isdigit` | Equivalente a la función integrada `str.isdigit`. |
| `islower` | Equivalente a la función integrada `str.islower`. |
| `isnumeric` | Equivalente a la función integrada `str.isnumeric`. |
| `isupper` | Equivalente a la función integrada `str.isupper`. |
| `join` | Une las cadenas en cada elemento de la Serie con el separador pasado. |
| `len` | Calcula la longitud de cada cadena. |
| `lower`, `upper` | Convierte el caso; equivalente a `x.lower()` o `x.upper()` para cada elemento. |
| `match` | Utiliza `re.match` con la expresión regular pasada en cada elemento, devolviendo `True` o `False` según si coincide. |
| `pad` | Añade espacios en blanco a la izquierda, derecha o ambos lados de las cadenas. |
| `center` | Equivalente a `pad(side="both")`. |
| `repeat` | Duplica los valores (por ejemplo, `s.str.repeat(3)` es equivalente a `x * 3` para cada cadena). |
| `replace` | Reemplaza las ocurrencias del patrón/expresión regular con alguna otra cadena. |
| `slice` | Extrae una parte (rebanada) de cada cadena en la Serie. |
| `split` | Divide las cadenas según un delimitador o expresión regular. |
| `strip` | Recorta los espacios en blanco de ambos lados, incluidos los saltos de línea. |
| `rstrip` | Recorta los espacios en blanco del lado derecho. |
| `lstrip` | Recorta los espacios en blanco del lado izquierdo. |

## 7.5 Categorical Data
### Background and Motivation
Many data systems (for data warehousing, statistical computing, or other uses) have developed specialized approaches for representing data with repeated values for more efficient storage and computation. In data warehousing, a best practice is to use so-called dimension tables containing the distinct values and storing the primary observations as integer keys referencing the dimension table:

In [101]:
values = pd.Series(['apple', 'orange', 'apple',
                    'apple'] * 2)
values
pd.unique(values)
pd.value_counts(values)

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
dtype: object

array(['apple', 'orange'], dtype=object)

C:\Users\ferch\AppData\Local\Temp\ipykernel_7288\3503291848.py:5: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(values)


apple     6
orange    2
Name: count, dtype: int64

In [102]:
values = pd.Series([0, 1, 0, 0] * 2)
dim = pd.Series(['apple', 'orange'])
values
dim

0    0
1    1
2    0
3    0
4    0
5    1
6    0
7    0
dtype: int64

0     apple
1    orange
dtype: object

In [103]:
dim.take(values)

0     apple
1    orange
0     apple
0     apple
0     apple
1    orange
0     apple
0     apple
dtype: object

In [104]:
fruits = ['apple', 'orange', 'apple', 'apple'] * 2
N = len(fruits)
rng = np.random.default_rng(seed=12345)
df = pd.DataFrame({'fruit': fruits,
                   'basket_id': np.arange(N),
                   'count': rng.integers(3, 15, size=N),
                   'weight': rng.uniform(0, 4, size=N)},
                  columns=['basket_id', 'fruit', 'count', 'weight'])
df

,basket_id,fruit,count,weight
0,0,apple,11,1.564438
1,1,orange,5,1.331256
2,2,apple,12,2.393235
3,3,apple,6,0.746937
4,4,apple,5,2.691024
5,5,orange,12,3.767211
6,6,apple,10,0.992983
7,7,apple,11,3.795525


In [105]:
fruit_cat = df['fruit'].astype('category')
fruit_cat

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

pandas has a special Categorical extension type for holding data that uses the integer-based categorical representation or encoding. This is a popular data compression technique for data with many occurrences of similar values and can provide significantly faster performance with lower memory use, especially for string data.

The values for fruit_cat are now an instance of pandas.Categorical, which you can access via the .array attribute:

In [106]:
c = fruit_cat.array
type(c)

pandas.core.arrays.categorical.Categorical

In [107]:
c.categories
c.codes

Index(['apple', 'orange'], dtype='object')

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

In [108]:
# A useful trick to get a mapping between codes and categories is:
dict(enumerate(c.categories))

{0: 'apple', 1: 'orange'}

In [109]:
df['fruit'] = df['fruit'].astype('category')
df["fruit"]

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

In [110]:
my_categories = pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])
my_categories

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, object): ['bar', 'baz', 'foo']

In [111]:
# If you have obtained categorical encoded data from another source, you can use the alternative from_codes constructor
categories = ['foo', 'bar', 'baz']
codes = [0, 1, 2, 0, 0, 1]
my_cats_2 = pd.Categorical.from_codes(codes, categories)
my_cats_2

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo', 'bar', 'baz']

In [112]:
ordered_cat = pd.Categorical.from_codes(codes, categories,
                                        ordered=True)
ordered_cat

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

In [113]:
my_cats_2.as_ordered()

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

In [114]:
rng = np.random.default_rng(seed=12345)
draws = rng.standard_normal(1000)
draws[:5]

array([-1.4238,  1.2637, -0.8707, -0.2592, -0.0753])

In [115]:
bins = pd.qcut(draws, 4)
bins

[(-3.121, -0.675], (0.687, 3.211], (-3.121, -0.675], (-0.675, 0.0134], (-0.675, 0.0134], ..., (0.0134, 0.687], (0.0134, 0.687], (-0.675, 0.0134], (0.0134, 0.687], (-0.675, 0.0134]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.121, -0.675] < (-0.675, 0.0134] < (0.0134, 0.687] < (0.687, 3.211]]

In [116]:
bins = pd.qcut(draws, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
bins
bins.codes[:10]

['Q1', 'Q4', 'Q1', 'Q2', 'Q2', ..., 'Q3', 'Q3', 'Q2', 'Q3', 'Q2']
Length: 1000
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

array([0, 3, 0, 1, 1, 0, 0, 2, 2, 0], dtype=int8)

In [117]:
bins = pd.Series(bins, name='quartile')
results = (pd.Series(draws)
           .groupby(bins)
           .agg(['count', 'min', 'max'])
           .reset_index())
results

C:\Users\ferch\AppData\Local\Temp\ipykernel_7288\2483392743.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(bins)


,quartile,count,min,max
0,Q1,250,-3.119609,-0.678494
1,Q2,250,-0.673305,0.008009
2,Q3,250,0.018753,0.686183
3,Q4,250,0.688282,3.211418


In [118]:
results['quartile']

0    Q1
1    Q2
2    Q3
3    Q4
Name: quartile, dtype: category
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [119]:
N = 10_000_000
labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))

In [120]:
categories = labels.astype('category')

In [121]:
labels.memory_usage(deep=True)
categories.memory_usage(deep=True)

600000128

10000540

In [122]:
%time _ = labels.astype('category')

CPU times: total: 312 ms
Wall time: 303 ms


In [123]:
%timeit labels.value_counts()
%timeit categories.value_counts()

239 ms ± 3.72 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
27.3 ms ± 265 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [124]:
s = pd.Series(['a', 'b', 'c', 'd'] * 2)
cat_s = s.astype('category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [125]:
cat_s.cat.codes
cat_s.cat.categories

0    0
1    1
2    2
3    3
4    0
5    1
6    2
7    3
dtype: int8

Index(['a', 'b', 'c', 'd'], dtype='object')

In [126]:
actual_categories = ['a', 'b', 'c', 'd', 'e']
cat_s2 = cat_s.cat.set_categories(actual_categories)
cat_s2

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (5, object): ['a', 'b', 'c', 'd', 'e']

In [127]:
cat_s.value_counts()
cat_s2.value_counts()

a    2
b    2
c    2
d    2
Name: count, dtype: int64

a    2
b    2
c    2
d    2
e    0
Name: count, dtype: int64

In [128]:
cat_s3 = cat_s[cat_s.isin(['a', 'b'])]
cat_s3
cat_s3.cat.remove_unused_categories()

0    a
1    b
4    a
5    b
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

0    a
1    b
4    a
5    b
dtype: category
Categories (2, object): ['a', 'b']

### Creating dummy variables for modeling
When you’re using statistics or machine learning tools, you’ll often transform categorical data into dummy variables, also known as one-hot encoding. This involves creating a DataFrame with a column for each distinct category; these columns contain 1s for occurrences of a given category and 0 otherwise. Consider the previous example. 

As mentioned previously in this chapter, the pandas.get_dummies function converts this one-dimensional categorical data into a DataFrame containing the dummy variable:

In [129]:
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2, dtype='category')

In [130]:
pd.get_dummies(cat_s, dtype=float)

,a,b,c,d
0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0
2,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0
5,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,1.0


In [131]:
pd.options.display.max_rows = PREVIOUS_MAX_ROWS